In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from pathlib import Path
from itertools import combinations

data_dir = Path('../data')
cache_dir = data_dir / 'cache'
external_dir = data_dir / 'external'
processed_dir = data_dir / 'processed'

In [2]:
def build_links_table(
    afiliations: pd.DataFrame,
    unis_cat: pd.DataFrame,
    domain: str = 'all',
    year_from: int | None = None,
    year_to: int | None = None,
    duplicate_bidirectional: bool = True,
) -> pd.DataFrame:
    """
    Build a co-affiliation links table from an authorships/affiliations DataFrame.

    Pairs are always stored in canonical (alphabetically sorted acronym) order,
    so 'UB'–'UAB' and 'UAB'–'UB' collapse to a single row: institution_a='UAB',
    institution_b='UB'.

    Parameters
    ----------
    afiliations : pd.DataFrame
        Must contain columns: 'work_openalex_id', 'institution_id',
        'year', 'domain_name'.
    unis_cat : pd.DataFrame
        Catalan universities with columns 'openalex_id', 'name', 'type', 'acronym'.
    domain : str
        Domain name to filter on. Use 'all' (case-insensitive) to skip the filter.
    year_from : int or None
        Inclusive lower bound on 'year'. None means no lower bound.
    year_to : int or None
        Inclusive upper bound on 'year'. None means no upper bound.
    duplicate_bidirectional : bool
        If True, also emit the swapped (B-A) row for every canonical (A-B) row.

    Returns
    -------
    pd.DataFrame
        Columns: 'institution_a', 'institution_b', 'weight'
        (acronyms in the institution columns, canonical order).
    """
    # ID -> acronym lookup
    id_to_acronym = dict(zip(unis_cat['openalex_id'], unis_cat['acronym']))

    df = afiliations

    # Year filter
    if year_from is not None or year_to is not None:
        if 'year' not in df.columns:
            raise ValueError("Year filter requested but 'year' column is missing.")
        if year_from is not None:
            df = df[df['year'] >= year_from]
        if year_to is not None:
            df = df[df['year'] <= year_to]

    # Domain filter
    if isinstance(domain, str) and domain.lower() != 'all':
        if 'domain_name' not in df.columns:
            raise ValueError("Domain filter requested but 'domain_name' column is missing.")
        df = df[df['domain_name'] == domain]

    # Build pairwise combinations per work (IDs already sorted)
    links = (
        df[['work_openalex_id', 'institution_id']]
        .drop_duplicates()
        .groupby('work_openalex_id')['institution_id']
        .apply(lambda s: list(combinations(sorted(s), 2)))
        .explode()
        .dropna()
        .value_counts()
        .reset_index(name='weight')
    )

    # Split tuples into two ID columns
    links[['institution_a', 'institution_b']] = pd.DataFrame(
        links['institution_id'].tolist(), index=links.index
    )
    links = links.drop(columns='institution_id')

    # Map IDs -> acronyms
    links['institution_a'] = links['institution_a'].map(id_to_acronym)
    links['institution_b'] = links['institution_b'].map(id_to_acronym)

    # Enforce canonical order on the acronyms themselves.
    # (Sorting by ID is not the same as sorting by acronym, so we re-sort here.)
    swap = links['institution_a'] > links['institution_b']
    links.loc[swap, ['institution_a', 'institution_b']] = (
        links.loc[swap, ['institution_b', 'institution_a']].values
    )

    # If ID sorting collapsed different-ID-equal-acronym rows into duplicates,
    # merge them back together.
    links = (
        links.groupby(['institution_a', 'institution_b'], as_index=False)['weight']
        .sum()
    )

    # # Optionally duplicate in both directions
    # if duplicate_bidirectional:
    #     links_rev = links.rename(
    #         columns={'institution_a': 'institution_b', 'institution_b': 'institution_a'}
    #     )
    #     links = pd.concat([links, links_rev], ignore_index=True)

    return links

In [3]:
data = [
    # Public (7)
    {'openalex_id': 'I71999127',   'name': 'Universitat de Barcelona',                              'type': 'Public',  'acronym': 'UB'},
    {'openalex_id': 'I123044942',  'name': 'Universitat Autònoma de Barcelona',                      'type': 'Public',  'acronym': 'UAB'},
    {'openalex_id': 'I9617848',    'name': 'Universitat Politècnica de Catalunya',                   'type': 'Public',  'acronym': 'UPC'},
    {'openalex_id': 'I170486558',  'name': 'Universitat Pompeu Fabra',                               'type': 'Public',  'acronym': 'UPF'},
    {'openalex_id': 'I251424209',  'name': 'Universitat de Girona',                                  'type': 'Public',  'acronym': 'UdG'},
    {'openalex_id': 'I15766328',   'name': 'Universitat de Lleida',                                  'type': 'Public',  'acronym': 'UdL'},
    {'openalex_id': 'I55952717',   'name': 'Universitat Rovira i Virgili',                           'type': 'Public',  'acronym': 'URV'},

    # Private (4)
    {'openalex_id': 'I5593406',    'name': 'Universitat Ramon Llull',                                'type': 'Private', 'acronym': 'URL'},
    {'openalex_id': 'I115304662',  'name': 'Universitat de Vic-Universitat Central de Catalunya',    'type': 'Private', 'acronym': 'UVic-UCC'},
    {'openalex_id': 'I3123915005', 'name': 'Universitat Internacional de Catalunya',                 'type': 'Private', 'acronym': 'UIC'},
    {'openalex_id': 'I2801339427', 'name': 'Universitat Abat Oliba CEU',                             'type': 'Private', 'acronym': 'UAO-CEU'},

    # Online (1)
    {'openalex_id': 'I138847295',  'name': 'Universitat Oberta de Catalunya',                        'type': 'Online',  'acronym': 'UOC'},
]

unis_cat = pd.DataFrame(data, columns=['openalex_id', 'name', 'type', 'acronym'])

unis_cat

,openalex_id,name,type,acronym
0,I71999127,Universitat de Barcelona,Public,UB
1,I123044942,Universitat Autònoma de Barcelona,Public,UAB
2,I9617848,Universitat Politècnica de Catalunya,Public,UPC
3,I170486558,Universitat Pompeu Fabra,Public,UPF
4,I251424209,Universitat de Girona,Public,UdG
5,I15766328,Universitat de Lleida,Public,UdL
6,I55952717,Universitat Rovira i Virgili,Public,URV
7,I5593406,Universitat Ramon Llull,Private,URL
8,I115304662,Universitat de Vic-Universitat Central de Cata...,Private,UVic-UCC
9,I3123915005,Universitat Internacional de Catalunya,Private,UIC


In [4]:
# Import complete df
afiliations = pd.read_parquet(external_dir / 'work_authorships.parquet')

# Reduce cols
afiliations = afiliations[['work_openalex_id', 'author_id', 'institution_id']].copy()

# Remove nan
afiliations = afiliations.dropna().reset_index(drop=True)

# Remove non catalan entries
catalan_ids = unis_cat['openalex_id'].tolist()
afiliations = afiliations[afiliations['institution_id'].isin(catalan_ids)].reset_index(drop=True)

afiliations

,work_openalex_id,author_id,institution_id
0,W1993308435,A5039571243,I71999127
1,W1993308435,A5064177201,I71999127
2,W2081673870,A5003175893,I55952717
3,W2081673870,A5067149287,I55952717
4,W2081673870,A5052548663,I55952717
...,...,...,...
696431,W1570020663,A5007639805,I9617848
696432,W1969633650,A5082431864,I123044942
696433,W1590121865,A5025955705,I71999127
696434,W1590121865,A5149372585,I71999127


In [5]:
# 1. Read data
works = pd.read_parquet(external_dir / 'works.parquet')
papers_cleaned_yr = pd.read_parquet(processed_dir / 'all_publications_enriched.parquet')

# 2. Build lookups (one row per key)
id2doi = works.dropna(subset=['openalex_id']).drop_duplicates(subset='openalex_id').set_index('openalex_id')['doi']

doi2meta = (
    papers_cleaned_yr
    .dropna(subset=['doi'])
    .drop_duplicates(subset='doi', keep='first')
    .set_index('doi')[['year', 'field_name', 'domain_name']]
)

# 3. Attach doi -> year/field/domain to afiliations
afiliations = afiliations.copy()
afiliations['doi'] = afiliations['work_openalex_id'].map(id2doi)
afiliations = afiliations.merge(doi2meta, on='doi', how='left')
afiliations = afiliations.drop(columns='doi')

# 4. Cast year to nullable int
afiliations['year'] = afiliations['year'].astype('Int64')

afiliations

,work_openalex_id,author_id,institution_id,year,field_name,domain_name
0,W1993308435,A5039571243,I71999127,1987,Chemistry,Physical Sciences
1,W1993308435,A5064177201,I71999127,1987,Chemistry,Physical Sciences
2,W2081673870,A5003175893,I55952717,1987,Materials Science,Physical Sciences
3,W2081673870,A5067149287,I55952717,1987,Materials Science,Physical Sciences
4,W2081673870,A5052548663,I55952717,1987,Materials Science,Physical Sciences
...,...,...,...,...,...,...
696431,W1570020663,A5007639805,I9617848,2005,Computer Science,Physical Sciences
696432,W1969633650,A5082431864,I123044942,2005,Medicine,Health Sciences
696433,W1590121865,A5025955705,I71999127,2014,Psychology,Social Sciences
696434,W1590121865,A5149372585,I71999127,2014,Psychology,Social Sciences


In [6]:
links = build_links_table(
    afiliations=afiliations,
    unis_cat=unis_cat,
    domain='all',
    year_from=2006,
)
links.to_csv(processed_dir / 'links_all.csv', index=False)

In [7]:
links = build_links_table(
    afiliations=afiliations,
    unis_cat=unis_cat,
    domain='Physical Sciences',
    year_from=2006,
)
links.to_csv(processed_dir / 'links_ps.csv', index=False)

In [8]:
links = build_links_table(
    afiliations=afiliations,
    unis_cat=unis_cat,
    domain='Life Sciences',
    year_from=2006,
)
links.to_csv(processed_dir / 'links_ls.csv', index=False)

In [9]:
links = build_links_table(
    afiliations=afiliations,
    unis_cat=unis_cat,
    domain='Social Sciences',
    year_from=2006,
)
links.to_csv(processed_dir / 'links_ss.csv', index=False)

In [10]:
links = build_links_table(
    afiliations=afiliations,
    unis_cat=unis_cat,
    domain='Health Sciences',
    year_from=2006,
)
links.to_csv(processed_dir / 'links_hs.csv', index=False)